# MAI-IR — лабораторные работы 1–5 в Google Colab

Ноутбук подготовлен для корпуса по **Warhammer 40,000**. ЛР №1 собирает статьи из нескольких MediaWiki-источников (Lexicanum, Warhammer 40,000 Wiki и русская Википедия), очищает тексты и проверяет требование **не менее 30 000 документов**. Затем ЛР №2–5 используют тот же `corpus.jsonl`.

**Важно:** сбор 30 000 документов может занять продолжительное время и зависит от доступности внешних API. Ячейки выполняют реальные запросы только при запуске в Colab; в этом файле заранее не заявляется, что корпус уже скачан. Соблюдайте правила источников и не публикуйте большие копии текстов без проверки лицензий.

Запускайте ячейки по порядку. Перед началом заполните ФИО, группу и контактный адрес. Для первого теста предусмотрен небольшой пробный сбор; полный сбор запускается отдельной ячейкой.

## 1. Загрузка архивов

In [ ]:
from google.colab import files
uploaded = files.upload()  # выберите lab01.zip ... lab05.zip и warhammer40k_corpus_collector.zip
print("Загружено файлов:", *uploaded.keys(), sep="\n- ")

In [ ]:
from pathlib import Path
import zipfile, shutil, os, json, subprocess, sys, textwrap

WORK = Path("/content/mai-ir")
WORK.mkdir(parents=True, exist_ok=True)
LAB_ZIPS = [f"lab{i:02d}.zip" for i in (1,2,3,4,5)]
COLLECTOR_ZIP = "warhammer40k_corpus_collector.zip"
required = LAB_ZIPS + [COLLECTOR_ZIP]
missing = [n for n in required if not Path("/content", n).exists()]
if missing:
    raise FileNotFoundError(f"Не найдены архивы: {missing}. Загрузите их в предыдущей ячейке.")

for name in LAB_ZIPS:
    with zipfile.ZipFile(Path("/content", name)) as zf:
        zf.extractall(WORK)
collector_root = WORK / "warhammer40k_corpus_collector"
collector_root.mkdir(exist_ok=True)
with zipfile.ZipFile(Path("/content", COLLECTOR_ZIP)) as zf:
    zf.extractall(WORK)
# В архиве сборщика скрипты лежат в warhammer40k_corpus/.
COLLECTOR = WORK / "warhammer40k_corpus"
assert (COLLECTOR / "collect_warhammer_corpus.py").exists(), f"Сборщик не найден в {COLLECTOR}"
print("Рабочий каталог:", WORK)
print("Сборщик:", COLLECTOR)
print("Лаборатории:", [p.name for p in sorted(WORK.glob("lab[0-9][0-9]"))])

## 2. Настройки

In [ ]:
# Заполните реальные данные перед созданием отчётов.
STUDENT = "ФИО студента"
GROUP = "Номер группы"
CATEGORY = "Warhammer 40,000 — корпус из нескольких энциклопедических источников"
CONTACT = "your-email@example.com"

# Сначала пробный сбор, затем полный. 30 000 — целевой объём после очистки.
SMOKE_TARGET = 300
TARGET_DOCS = 30000
MIN_DOC_CHARS = 200
CRAWL_DEPTH = 5

assert STUDENT != "ФИО студента", "Заполните STUDENT"
assert GROUP != "Номер группы", "Заполните GROUP"
assert "@" in CONTACT and CONTACT != "your-email@example.com", "Укажите рабочий контактный e-mail"

## 3. Системные зависимости

In [ ]:
import subprocess
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "g++", "make", "pandoc", "libreoffice"], check=True)
subprocess.run([sys.executable, "--version"], check=True)
subprocess.run(["g++", "--version"], check=True, stdout=subprocess.DEVNULL)

## 4. ЛР №1 — пробный сбор корпуса Warhammer 40,000

Сначала собираем небольшую выборку в отдельную папку, чтобы проверить доступность API и качество очистки. Пробный результат не считается итоговым корпусом.

In [ ]:
LAB1 = WORK / "lab01"
LAB1_DATA = LAB1 / "data"
SMOKE_DATA = LAB1 / "data_smoke"
SMOKE_DATA.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, str(COLLECTOR / "collect_warhammer_corpus.py"),
       "--out", str(SMOKE_DATA), "--target", str(SMOKE_TARGET),
       "--contact", CONTACT, "--depth", str(CRAWL_DEPTH),
       "--min-chars", str(MIN_DOC_CHARS)]
print("Команда:", " ".join(cmd))
subprocess.run(cmd, cwd=COLLECTOR, check=True)
smoke_stats = json.loads((SMOKE_DATA / "stats.json").read_text(encoding="utf-8"))
print("Пробный сбор:", smoke_stats.get("docs", 0), "очищенных документов")
print("Источники:", smoke_stats.get("by_source", {}))

## 5. ЛР №1 — полный корпус (цель: 30 000 документов)

Эта ячейка может работать долго. Сборщик сохраняет промежуточные результаты в `raw.jsonl`, поэтому при сетевом сбое ячейку можно запустить повторно. После завершения проверяется **фактическое** количество документов. Если какой-либо API недоступен или источники не дают достаточного числа статей, статистика явно покажет недобор — не указывайте в отчёте, что требование выполнено, если `requirement_met` равно `false`.

In [ ]:
LAB1_DATA.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, str(COLLECTOR / "collect_warhammer_corpus.py"),
       "--out", str(LAB1_DATA), "--target", str(TARGET_DOCS),
       "--contact", CONTACT, "--depth", str(CRAWL_DEPTH),
       "--min-chars", str(MIN_DOC_CHARS)]
print("Запускаю полный сбор. Цель:", TARGET_DOCS)
subprocess.run(cmd, cwd=COLLECTOR, check=True)
collector_stats_path = LAB1_DATA / "stats.json"
collector_stats = json.loads(collector_stats_path.read_text(encoding="utf-8"))
# Сохраняем специфическую статистику сборщика отдельно; ниже будет рассчитана статистика формата ЛР1.
(LAB1_DATA / "collector_stats.json").write_text(json.dumps(collector_stats, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(collector_stats, ensure_ascii=False, indent=2))
if not collector_stats.get("requirement_met", False):
    print("ВНИМАНИЕ: порог 30 000 очищенных документов пока НЕ достигнут. Можно повторить сбор или расширить источники/категории.")

### Статистика и отчёт ЛР №1

Пересчитываем метрики штатным инструментом лабораторной работы. Исходные тексты `raw.jsonl` сохраняются, а `collector_stats.json` хранит статистику самого сборщика.

In [ ]:
# Подготовим метаданные в формате отчёта ЛР1.
meta = {
    "category": CATEGORY,
    "depth": CRAWL_DEPTH,
    "sources": [
        "https://wh40k.lexicanum.com/wiki/Main_Page",
        "https://warhammer40k.wiki.gg/wiki/Main_Page",
        "https://ru.wikipedia.org/wiki/Warhammer_40,000"
    ],
    "collector": "collect_warhammer_corpus.py",
    "target_docs": TARGET_DOCS
}
(LAB1_DATA / "fetch_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
subprocess.run([sys.executable, str(LAB1 / "corpus_stats.py"), str(LAB1_DATA / "corpus.jsonl"),
                "--raw", str(LAB1_DATA / "raw.jsonl"), "--min-docs", str(TARGET_DOCS),
                "-o", str(LAB1_DATA / "stats.json")], check=True)
subprocess.run([sys.executable, str(LAB1 / "make_report.py"), "--stats", str(LAB1_DATA / "stats.json"),
                "--meta", str(LAB1_DATA / "fetch_meta.json"), "--student", STUDENT, "--group", GROUP,
                "--min-chars", str(MIN_DOC_CHARS), "-o", str(LAB1 / "report.md")], cwd=LAB1, check=True)
subprocess.run(["bash", "build_report.sh"], cwd=LAB1, check=True)
final_stats = json.loads((LAB1_DATA / "stats.json").read_text(encoding="utf-8"))
print("Документов:", final_stats["docs"])
print("Требование 30 000 выполнено:", final_stats["requirements"]["docs_ok"])
print("Файлы:", [p.name for p in [LAB1_DATA/"raw.jsonl", LAB1_DATA/"corpus.jsonl", LAB1_DATA/"stats.json", LAB1/"report.docx", LAB1/"report.pdf"] if p.exists()])

## 6. ЛР №1 — автотесты

Тесты проверяют очистку и конвейер ЛР1. Они не скачивают корпус.

In [ ]:
subprocess.run(["bash", "run_tests.sh"], cwd=LAB1, check=True)
print((LAB1 / "tests_result.txt").read_text(encoding="utf-8", errors="replace") if (LAB1 / "tests_result.txt").exists() else "Тесты завершились без ошибки.")

## 7. ЛР №2 — поисковый робот

Тесты можно запускать без MongoDB. Для пробного обхода используется режим `memory`; полноценный режим лабораторной, если он требует постоянного хранилища, выполняйте с доступной MongoDB. Не запускайте массовый обход без согласования нагрузки и ограничений сайтов.

In [ ]:
LAB2 = WORK / "lab02"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=LAB2, check=True)
subprocess.run(["bash", "run_tests.sh"], cwd=LAB2, check=True)

In [ ]:
corpus_path = LAB1_DATA / "corpus.jsonl"
if not corpus_path.exists():
    raise FileNotFoundError("Сначала завершите сбор корпуса в ЛР1.")
subprocess.run([sys.executable, "crawl.py", str(corpus_path), "--contact", CONTACT,
                "--store", "memory", "--limit", "20"], cwd=LAB2, check=True)

### Полноценный запуск ЛР №2 с MongoDB (по необходимости)

В Colab MongoDB-сервер обычно не установлен. Если задание требует MongoDB, настройте доступный сервер и передайте URI через секрет/переменную окружения `MONGO_URI`, не публикуя пароль. Затем изучите `lab02/README.md` и запустите предусмотренный проектом `run_all.sh`. Режим `memory` выше является только пробным обходом.

## 8. ЛР №3 — токенизация

In [ ]:
LAB3 = WORK / "lab03"
subprocess.run(["bash", "run_tests.sh"], cwd=LAB3, check=True)
subprocess.run(["bash", "run_all.sh", str(corpus_path), STUDENT, GROUP, CATEGORY], cwd=LAB3, check=True)
tokens_path = LAB3 / "data" / "tokens.txt"
print("Файл токенов:", tokens_path, "размер:", tokens_path.stat().st_size if tokens_path.exists() else "не создан")

## 9. ЛР №4 — лемматизация и стемминг

ЛР4 использует токены, полученные ЛР3. Нормализованные результаты сохраняются отдельно от исходных токенов.

In [ ]:
LAB4 = WORK / "lab04"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=LAB4, check=True)
subprocess.run(["bash", "run_tests.sh"], cwd=LAB4, check=True)
subprocess.run(["bash", "run_all.sh", str(tokens_path), STUDENT, GROUP, CATEGORY], cwd=LAB4, check=True)
print("Результаты ЛР4:", [p.name for p in (LAB4/"data").glob("*")])

## 10. ЛР №5 — закон Ципфа

In [ ]:
LAB5 = WORK / "lab05"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], cwd=LAB5, check=True)
subprocess.run(["bash", "run_tests.sh"], cwd=LAB5, check=True)
subprocess.run(["bash", "run_all.sh", str(tokens_path), STUDENT, GROUP, CATEGORY], cwd=LAB5, check=True)
print("Результаты ЛР5:", [p.name for p in (LAB5/"data").glob("*")])

## 11. Сохранение результатов в Google Drive

Подключите Google Drive и скопируйте лабораторные вместе с корпусом, исходниками, статистикой, тестами и отчётами.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DEST = Path('/content/drive/MyDrive/MAI-IR-Warhammer40k')
DEST.mkdir(parents=True, exist_ok=True)
for lab_name in ["lab01", "lab02", "lab03", "lab04", "lab05"]:
    src = WORK / lab_name
    dst = DEST / lab_name
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
# Копируем сборщик и его README для воспроизводимости.
collector_dst = DEST / "warhammer40k_corpus"
if collector_dst.exists():
    shutil.rmtree(collector_dst)
shutil.copytree(COLLECTOR, collector_dst, ignore=shutil.ignore_patterns("__pycache__"))
print("Скопировано в:", DEST)

## 12. Финальная проверка перед сдачей

- [ ] `lab01/data/corpus.jsonl` содержит минимум 30 000 очищенных документов; проверено `lab01/data/stats.json → requirements.docs_ok`.
- [ ] Сохранены `raw.jsonl`, `collector_stats.json`, `stats.json`, `fetch_meta.json` и источники.
- [ ] ЛР1 отчёт DOCX/PDF создан, в журнале выполнения указаны фактические действия и дата.
- [ ] ЛР2: режим хранения и ограничения обхода описаны честно; если требуется MongoDB, выполнен соответствующий запуск.
- [ ] ЛР3: токены, словарь, статистика, проблемные токены и бенчмарк созданы.
- [ ] ЛР4: леммы, стеммы, сравнение и отчёт созданы.
- [ ] ЛР5: частотный анализ, графики и отчёт созданы.
- [ ] Во всех отчётах указаны реальные ФИО, группа, тема и фактические результаты; никаких неподтверждённых значений.
- [ ] Проверены правила использования и лицензии источников перед передачей/публикацией корпуса.

**Примечание:** сборщик может не достичь 30 000 документов при недоступности API или ограничениях источников. В этом случае расширьте категории/источники или повторите сбор и не отмечайте требование выполненным до фактической проверки.